# ARC Prize 2026 (ARC-AGI-2) — offline submission

Forks the 2025 winner **NVARC** (Sorokin & Puget): reuses their public fine-tuned
checkpoint `qwen3_4b_grids15_sft139` and runs **test-time training -> turbo_dfs ->
augmentation-NLL voting** on Kaggle L4x4, offline, in a `transformers`+`peft` port
(no unsloth). ~240 tasks / 4 GPUs @ ~325s/task (aug_n=8) ~= 5.4h, within 12h.

**Before running, attach as notebook inputs:**
1. The model `sorokin/qwen3_4b_grids15_sft139` (Kaggle *Model* input).
2. This repo's `src/` folder as a Kaggle *Dataset* (e.g. `arc-agi2-nvarc-fork`).
3. The competition dataset `arc-prize-2026-arc-agi-2`.
Internet **off**.

In [ ]:
import sys, os, glob
# point at our code (attach src/ as a Kaggle dataset)
for c in ['/kaggle/input/arc-agi2-nvarc-fork/src', '/kaggle/input/arc-agi2-solver/src', 'src', '/kaggle/working/src']:
    if os.path.isdir(c):
        sys.path.insert(0, c); print('code:', c); break
from nvarc.kaggle_submit import main, find_test_challenges

# locate the SFT'd checkpoint (attached Kaggle Model)
model_dir = None
for c in glob.glob('/kaggle/input/**/config.json', recursive=True):
    if 'qwen3_4b_grids15' in c or 'qwen3_2b_grids15' in c:
        model_dir = os.path.dirname(c); break
print('model:', model_dir)
print('test :', find_test_challenges())

In [ ]:
# Full offline run -> /kaggle/working/submission.json
# Tunables: aug_n (TTFT augmentations; 8 fast / 16 = full NVARC), eval_aug_n (inference views),
#           per_task_budget (s), nprocs (defaults to GPU count).
main(
    model_dir=model_dir,
    out_path='/kaggle/working/submission.json',
    nprocs=None,
    aug_n=8,
    eval_aug_n=2,
    per_task_budget=600,
    reserve_s=1200,   # leave 20min for submission writing
)